In [1]:
import os
from pathlib import Path
import re

import numpy as np
import pandas as pd


## Add some data in manually

We would like to have monthly data that goes as far back as 1991. Unfortunately, before 1995, the data for some crops is in PDF form instead of text (and later, XLS) files. The data format also changes quite a bit over the years, both in naming conventions, where data is placed in a report (which interferes with regex search patterns), and more.

For now, we are going to add in some data manually *if* the crop is `cotton`, primarily the May data for 1991 - 1995. The official data the USDA uses in reports and figures comes out in May, so instead of adding in monthly data for these dates, we are only going to focus on annual data. The data for `corn` and `soybeans`, however, goes back to 1991.


In [2]:
CROP = 'cotton'
PROJECT_ROOT = Path('.').resolve().parent                     # .. project root
INPUT_DIR = PROJECT_ROOT / 'data' / 'raw' / f'{CROP}'         # e.g., ../data/raw/cotton/
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / f'{CROP}'  # e.g., ../data/processed/cotton/

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if CROP == 'corn':
    XLS_SEARCH_TERM = 'Feed Grain and Corn Supply and Use'
elif CROP == 'cotton':
    XLS_SEARCH_TERM = 'Cotton Supply and Use'
elif CROP == 'soybean':
    XLS_SEARCH_TERM = 'Soybeans and Products Supply and Use'
else:
    raise ValueError("`CROP` must be one of ['corn', 'cotton', 'soybean']")


In [3]:
if CROP == 'cotton':
    dates = [pd.to_datetime(date) for date in ['1991-05-01', '1992-05-01', '1993-05-01', '1994-05-01', '1995-05-01']]

    pa_lines = [
        [10.59, 12.35, np.nan, np.nan],
        [12.35, 14.05, np.nan, np.nan],
        [14.05, 13.24, np.nan, 13.43],
        [13.24, 13.44, np.nan, 13.84],
        [13.44, 13.73, np.nan, 16.20],
    ]
    
    ha_lines = [
        [9.54, 11.73, np.nan, np.nan],
        [11.73, 12.96, np.nan, np.nan],
        [12.96, 11.14, np.nan, 12.36],
        [11.14, 12.79, np.nan, 12.80],
        [12.78, 13.33, np.nan, 15.15],
    ]
    
    ya_lines = [
        [614, 634, np.nan, np.nan],
        [634, 652, np.nan, np.nan],
        [652, 699, np.nan, 680],
        [699, 606, np.nan, 665],
        [606, 708, np.nan, 665],
    ]

    df_manual = pd.DataFrame({
        # duplicate time column for a datetime index that will be dropped later
        'dates': dates,
        'time': dates,
        
        'year_minus_2_planted_area': [num[0] for num in pa_lines],
        'year_minus_1_planted_area': [num[1] for num in pa_lines],
        'current_year_last_month_planted_area': [num[2] for num in pa_lines],
        'current_year_current_month_planted_area': [num[3] for num in pa_lines],
    
        'year_minus_2_harvested_area': [num[0] for num in ha_lines],
        'year_minus_1_harvested_area': [num[1] for num in ha_lines],
        'current_year_last_month_harvested_area': [num[2] for num in ha_lines],
        'current_year_current_month_harvested_area': [num[3] for num in ha_lines],
    
        'year_minus_2_yield': [num[0] for num in ya_lines],
        'year_minus_1_yield': [num[1] for num in ya_lines],
        'current_year_last_month_yield': [num[2] for num in ya_lines],
        'current_year_current_month_yield': [num[3] for num in ya_lines],
    })
    
    df_manual = df_manual.set_index('dates')
    df_manual


## Read in and process text files

In [4]:
def convert_and_pad_list(input_list):
    """
    Given an input list, replace all string values with float equivalents.
    Then, make sure the list has 4 elements. If not, insert a NaN value at index 3.
    """
    input_list = [float(val) if val != 'na' else np.nan for val in input_list]
    if len(input_list) == 3:
        input_list.insert(2, np.nan)

    return input_list


In [5]:
txt_filenames = sorted(list(INPUT_DIR.glob('*.txt')))
for filename in txt_filenames[:10]:
    print(str(filename)[24:])
print()
for filename in txt_filenames[-10:]:
    print(str(filename)[24:])


usda-wasde-web-scraping/data/raw/cotton/cotton_1996_05.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_1996_06.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_1996_07.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_1996_08.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_1996_09.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_1996_10.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_1996_11.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_1996_12.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_1997_01.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_1997_02.txt

usda-wasde-web-scraping/data/raw/cotton/cotton_2025_04.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_2025_05.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_2025_06.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_2025_07.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_2025_08.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_2025_09.txt
usda-wasde-web-scraping/data/raw/cotton/cotton_2025_11.

In [6]:
dates = []
pa_lines = []
ha_lines = []
ya_lines = []

for filename in txt_filenames:
    # convert PosixPath to str
    filename = str(filename)
    
    year, month = filename[-11:-4].split('_')
    date = f"{year}-{month}-01"
    dates.append(pd.to_datetime(date))
    
    with open(filename) as f:
        lines = [line.lower().replace('\n', '').replace('*', '').replace(':', '').lstrip() for line in f]

        if CROP == 'cotton':
            matching_lines = [line for line in lines if 'planted' in line]
            pa_line = matching_lines[0].replace('planted', '').split()
            pa_line = convert_and_pad_list(pa_line)
            pa_lines.append(pa_line)
    
            matching_lines = [line for line in lines if 'harvested' in line]
            ha_line = matching_lines[0].replace('harvested', '').split()
            ha_line = convert_and_pad_list(ha_line)
            ha_lines.append(ha_line)
        
            # the label for the yield row changes in June 1998 
            if pd.to_datetime(f'{year}-{month}-01') < pd.to_datetime('1998-06-01'):
                matching_lines = [line for line in lines if 'yield per harv. acre' in line]
                ya_line = matching_lines[0].replace('yield per harv. acre', '').split()
            else:
                pattern = re.compile(r'^.*\bacre\b.*$', re.MULTILINE | re.IGNORECASE)
                matching_lines = [line for line in lines if pattern.search(line)]
                ya_line = matching_lines[0].replace('acre', '').split()

            ya_line = convert_and_pad_list(ya_line)
            ya_lines.append(ya_line)
        
        elif CROP == 'soybean':
            matching_lines = [line for line in lines if 'planted' in line]
            pa_line = matching_lines[0].replace('area', '').replace('planted', '').split()
            pa_line = convert_and_pad_list(pa_line)
            pa_lines.append(pa_line)
    
            matching_lines = [line for line in lines if 'harvested' in line]
            ha_line = matching_lines[0].replace('area', '').replace('harvested', '').split()
            ha_line = convert_and_pad_list(ha_line)
            ha_lines.append(ha_line)
        
            # the label for the yield row changes in June 1998 
            if pd.to_datetime(f'{year}-{month}-01') < pd.to_datetime('1996-05-01'):
                matching_lines = [line for line in lines if 'unit' in line]
                ya_line = matching_lines[0].replace('unit', '').split()
            elif pd.to_datetime(f'{year}-{month}-01') == pd.to_datetime('1996-05-01'):
                matching_lines = [line for line in lines if 'yield per harv. acre' in line]
                ya_line = matching_lines[0].replace('yield per harv. acre', '').split()
            elif pd.to_datetime(f'{year}-{month}-01') > pd.to_datetime('1996-05-01') and pd.to_datetime(f'{year}-{month}-01') < pd.to_datetime('2016-10-01'):
                pattern = re.compile(r'^.*\bacre\b.*$', re.MULTILINE | re.IGNORECASE)
                matching_lines = [line for line in lines if pattern.search(line)]
                ya_line = matching_lines[0].replace('acre', '').split()
            else: # date > 2016-10-01
                matching_lines = [line for line in lines if 'yield per harvested acre' in line]
                ya_line = matching_lines[0].replace('yield per harvested acre', '').split()

            ya_line = convert_and_pad_list(ya_line)
            ya_lines.append(ya_line)
        else:
            raise ValueError(f'Processing data for {CROP} is not currenty supported.')

print(len(dates))
print(len(pa_lines))
print(len(ha_lines))
print(len(ya_lines))


284
284
284
284


In [7]:
df_txt = pd.DataFrame({
    # duplicate time column for a datetime index that will be dropped later
    'dates': dates,
    'time': dates,
    
    'year_minus_2_planted_area': [num[0] for num in pa_lines],
    'year_minus_1_planted_area': [num[1] for num in pa_lines],
    'current_year_last_month_planted_area': [num[2] for num in pa_lines],
    'current_year_current_month_planted_area': [num[3] for num in pa_lines],

    'year_minus_2_harvested_area': [num[0] for num in ha_lines],
    'year_minus_1_harvested_area': [num[1] for num in ha_lines],
    'current_year_last_month_harvested_area': [num[2] for num in ha_lines],
    'current_year_current_month_harvested_area': [num[3] for num in ha_lines],

    'year_minus_2_yield': [num[0] for num in ya_lines],
    'year_minus_1_yield': [num[1] for num in ya_lines],
    'current_year_last_month_yield': [num[2] for num in ya_lines],
    'current_year_current_month_yield': [num[3] for num in ya_lines],
})

df_txt = df_txt.set_index('dates')
df_txt


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
dates,,,,,,,,,,,,,
1996-05-01,1996-05-01,13.72,16.93,NaN,15.25,13.32,16.01,NaN,14.00,708.0,537.0,NaN,650.0
1996-06-01,1996-06-01,13.72,16.93,15.25,15.25,13.32,16.01,14.00,14.00,708.0,537.0,650.0,650.0
1996-07-01,1996-07-01,13.72,16.93,15.25,14.36,13.32,16.01,14.00,13.71,708.0,537.0,650.0,665.0
1996-08-01,1996-08-01,13.72,16.93,14.36,14.24,13.32,16.01,13.71,12.99,708.0,537.0,665.0,686.0
1996-09-01,1996-09-01,13.72,16.93,14.24,14.24,13.32,16.01,12.99,12.99,708.0,537.0,686.0,661.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-01,2025-09-01,10.23,11.18,9.28,9.30,6.44,7.81,7.36,7.37,899.0,886.0,862.0,861.0
2025-11-01,2025-11-01,10.23,11.18,9.30,9.30,6.44,7.81,7.37,7.37,899.0,886.0,861.0,919.0
2025-12-01,2025-12-01,10.23,11.18,9.30,9.30,6.44,7.81,7.37,7.37,899.0,886.0,919.0,929.0


In [8]:
# print missing dates before having processed xls files
missing_dates = pd.date_range(start=df_txt.index.min(), end=df_txt.index.max(), freq='MS').difference(df_txt.index).tolist()
missing_dates = [date.strftime('%Y-%m-%d') for date in missing_dates]
print(missing_dates)


['2010-10-01', '2010-11-01', '2010-12-01', '2011-01-01', '2011-02-01', '2011-03-01', '2011-04-01', '2011-05-01', '2011-06-01', '2011-07-01', '2011-08-01', '2011-09-01', '2011-10-01', '2011-11-01', '2011-12-01', '2012-01-01', '2012-02-01', '2012-03-01', '2012-04-01', '2012-05-01', '2012-06-01', '2012-07-01', '2012-08-01', '2012-09-01', '2012-10-01', '2012-11-01', '2012-12-01', '2013-01-01', '2013-02-01', '2013-03-01', '2013-04-01', '2013-05-01', '2013-06-01', '2013-07-01', '2013-08-01', '2013-09-01', '2013-10-01', '2013-11-01', '2013-12-01', '2014-01-01', '2014-02-01', '2014-03-01', '2014-04-01', '2014-05-01', '2014-06-01', '2014-07-01', '2014-08-01', '2014-09-01', '2014-10-01', '2014-11-01', '2014-12-01', '2015-01-01', '2015-02-01', '2015-03-01', '2015-04-01', '2015-05-01', '2015-06-01', '2015-07-01', '2015-08-01', '2015-09-01', '2015-10-01', '2015-11-01', '2015-12-01', '2016-01-01', '2016-02-01', '2016-03-01', '2016-04-01', '2016-05-01', '2016-06-01', '2016-07-01', '2016-08-01', '2016

## Read in and process XLS files

In [9]:
xls_filenames = sorted(list(INPUT_DIR.glob('*.xls')))
for filename in xls_filenames[:10]:
    print(str(filename)[24:])
print()
for filename in xls_filenames[-10:]:
    print(str(filename)[24:])


usda-wasde-web-scraping/data/raw/cotton/cotton_2010_10.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2010_11.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2010_12.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2011_01.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2011_02.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2011_03.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2011_04.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2011_05.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2011_06.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2011_07.xls

usda-wasde-web-scraping/data/raw/cotton/cotton_2015_12.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2016_01.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2016_02.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2016_03.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2016_04.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2016_05.xls
usda-wasde-web-scraping/data/raw/cotton/cotton_2016_06.

In [10]:
dates = []
pa_lines = []
ha_lines = []
ya_lines = []

for filename in xls_filenames:
    # convert PosixPath to str
    filename = str(filename)
    
    year, month = filename[-11:-4].split('_')
    date = f"{year}-{month}-01"
    dates.append(pd.to_datetime(date))

    # test case files: cotton_2016_08 has a different format than cotton_2010_10
    df = pd.read_excel(filename)
    # drop filler columns and rows
    df = df[~df.apply(lambda row: row.astype(str).str.contains(XLS_SEARCH_TERM).any(), axis=1)]
    df = df[~df.apply(lambda row: row.astype(str).str.contains('WASDE').any(), axis=1)]
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)

    if CROP == 'cotton':
        df.columns = ['label', 'year_minus_2', 'year_minus_1', 'last_month', 'current_month']
    
        df['label'] = df.label.str.lower().str.strip()
        df = df.query(" label == 'planted' or label == 'harvested' or label == 'yield per harvested acre' ")
    
        # remove all asterisks
        for column in df.columns:
            df[column] = df[column].astype(str).str.replace('*', '')
        
        pa_line = df.query(" label == 'planted' ").iloc[:, 1:].values[0].tolist()
        ha_line = df.query(" label == 'harvested' ").iloc[:, 1:].values[0].tolist()
        ya_line = df.query(" label == 'yield per harvested acre' ").iloc[:, 1:].values[0].tolist()
        
        pa_line = convert_and_pad_list(pa_line)
        ha_line = convert_and_pad_list(ha_line)
        ya_line = convert_and_pad_list(ya_line)
    
        pa_lines.append(pa_line)
        ha_lines.append(ha_line)
        ya_lines.append(ya_line)

    elif CROP == 'soybean':
        if pd.to_datetime(date) < pd.to_datetime('2015-07-01'):
            df.columns = ['label', 'date', 'year_minus_2', 'year_minus_1', 'last_month', 'current_month']
        else:
            df.columns = ['label', 'year_minus_2', 'year_minus_1', 'last_month', 'current_month']

        df['label'] = df.label.str.lower().str.strip()
        df = df.query(" label == 'area planted' or label == 'area harvested' or label == 'yield per harvested acre' ")
        
        # remove all asterisks
        for column in df.columns:
            df[column] = df[column].astype(str).str.replace('*', '')
    
        # the soybean data has an extra 'date' column for some month-years, so we need to shift up a column in the values that we take
        if pd.to_datetime(date) < pd.to_datetime('2015-07-01'):
            pa_line = df.query(" label == 'area planted' ").iloc[:, 2:].values[0].tolist()
            ha_line = df.query(" label == 'area harvested' ").iloc[:, 2:].values[0].tolist()
            ya_line = df.query(" label == 'yield per harvested acre' ").iloc[:, 2:].values[0].tolist()
        else:
            pa_line = df.query(" label == 'area planted' ").iloc[:, 1:].values[0].tolist()
            ha_line = df.query(" label == 'area harvested' ").iloc[:, 1:].values[0].tolist()
            ya_line = df.query(" label == 'yield per harvested acre' ").iloc[:, 1:].values[0].tolist()
        
        pa_line = convert_and_pad_list(pa_line)
        ha_line = convert_and_pad_list(ha_line)
        ya_line = convert_and_pad_list(ya_line)
    
        pa_lines.append(pa_line)
        ha_lines.append(ha_line)
        ya_lines.append(ya_line)

print(len(dates))
print(len(pa_lines))
print(len(ha_lines))
print(len(ya_lines))


71
71
71
71


In [11]:
df_xls = pd.DataFrame({
    # duplicate time column for a datetime index that will be dropped later
    'dates': dates,
    'time': dates,
    
    'year_minus_2_planted_area': [num[0] for num in pa_lines],
    'year_minus_1_planted_area': [num[1] for num in pa_lines],
    'current_year_last_month_planted_area': [num[2] for num in pa_lines],
    'current_year_current_month_planted_area': [num[3] for num in pa_lines],

    'year_minus_2_harvested_area': [num[0] for num in ha_lines],
    'year_minus_1_harvested_area': [num[1] for num in ha_lines],
    'current_year_last_month_harvested_area': [num[2] for num in ha_lines],
    'current_year_current_month_harvested_area': [num[3] for num in ha_lines],

    'year_minus_2_yield': [num[0] for num in ya_lines],
    'year_minus_1_yield': [num[1] for num in ya_lines],
    'current_year_last_month_yield': [num[2] for num in ya_lines],
    'current_year_current_month_yield': [num[3] for num in ya_lines],
})

df_xls = df_xls.set_index('dates')
df_xls


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
dates,,,,,,,,,,,,,
2010-10-01,2010-10-01,9.47,9.15,11.04,11.04,7.57,7.53,10.77,10.77,813.0,777.0,839.0,841.0
2010-11-01,2010-11-01,9.47,9.15,11.04,11.04,7.57,7.53,10.77,10.77,813.0,777.0,841.0,821.0
2010-12-01,2010-12-01,9.47,9.15,11.04,11.04,7.57,7.53,10.77,10.77,813.0,777.0,821.0,814.0
2011-01-01,2011-01-01,9.47,9.15,11.04,10.97,7.57,7.53,10.77,10.71,813.0,777.0,814.0,821.0
2011-02-01,2011-02-01,9.47,9.15,10.97,10.97,7.57,7.53,10.71,10.71,813.0,777.0,821.0,821.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016-05-01,2016-05-01,11.04,8.58,NaN,9.56,9.35,8.07,NaN,8.80,838.0,766.0,NaN,807.0
2016-06-01,2016-06-01,11.04,8.58,9.56,9.56,9.35,8.07,8.80,8.80,838.0,766.0,807.0,807.0
2016-07-01,2016-07-01,11.04,8.58,9.56,10.02,9.35,8.07,8.80,9.30,838.0,766.0,807.0,815.0


## Merge processed dataframes for text and XLS files

In [12]:
# df_txt already has empty rows for the dates where there are xls files
# so we want to infill those while keeping df_txt the same
if CROP == 'cotton':
    df = pd.concat([df_manual, df_txt, df_xls]).sort_values(by='time')
else:
    df = pd.concat([df_txt, df_xls]).sort_values(by='time')

df


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
dates,,,,,,,,,,,,,
1991-05-01,1991-05-01,10.59,12.35,NaN,NaN,9.54,11.73,NaN,NaN,614.0,634.0,NaN,NaN
1992-05-01,1992-05-01,12.35,14.05,NaN,NaN,11.73,12.96,NaN,NaN,634.0,652.0,NaN,NaN
1993-05-01,1993-05-01,14.05,13.24,NaN,13.43,12.96,11.14,NaN,12.36,652.0,699.0,NaN,680.0
1994-05-01,1994-05-01,13.24,13.44,NaN,13.84,11.14,12.79,NaN,12.80,699.0,606.0,NaN,665.0
1995-05-01,1995-05-01,13.44,13.73,NaN,16.20,12.78,13.33,NaN,15.15,606.0,708.0,NaN,665.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-01,2025-09-01,10.23,11.18,9.28,9.30,6.44,7.81,7.36,7.37,899.0,886.0,862.0,861.0
2025-11-01,2025-11-01,10.23,11.18,9.30,9.30,6.44,7.81,7.37,7.37,899.0,886.0,861.0,919.0
2025-12-01,2025-12-01,10.23,11.18,9.30,9.30,6.44,7.81,7.37,7.37,899.0,886.0,919.0,929.0


In [13]:
full_date_list = pd.date_range(start=df.index.min(), end=df.index.max(), freq='MS')
df = df.reindex(full_date_list)
df['time'] = df.index

# now let's format the time column as a string
df = df.reset_index(drop=True)
df['time'] = df.time.dt.strftime('%Y-%m-%d')

df


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
0,1991-05-01,10.59,12.35,NaN,NaN,9.54,11.73,NaN,NaN,614.0,634.0,NaN,NaN
1,1991-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1991-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1991-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1991-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,2025-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
414,2025-11-01,10.23,11.18,9.30,9.30,6.44,7.81,7.37,7.37,899.0,886.0,861.0,919.0
415,2025-12-01,10.23,11.18,9.30,9.30,6.44,7.81,7.37,7.37,899.0,886.0,919.0,929.0
416,2026-01-01,10.23,11.18,9.30,9.28,6.44,7.81,7.37,7.80,899.0,886.0,929.0,856.0


In [14]:
# check for missing dates
data_cols = df.columns.values[1:].tolist()
all_nan_mask = df[data_cols].isna().all(axis=1)
found_missing_dates = df.loc[all_nan_mask, 'time'].tolist()
print(found_missing_dates)

known_missing_dates = ['2013-10-01', '2019-01-01', '2025-10-01']
# if this test passes, thn we have data for all but when we know it is missing
# because we only imported data for may in 1992-95, we can ignore those dates
assert(found_missing_dates[-3:] == known_missing_dates)


['1991-06-01', '1991-07-01', '1991-08-01', '1991-09-01', '1991-10-01', '1991-11-01', '1991-12-01', '1992-01-01', '1992-02-01', '1992-03-01', '1992-04-01', '1992-06-01', '1992-07-01', '1992-08-01', '1992-09-01', '1992-10-01', '1992-11-01', '1992-12-01', '1993-01-01', '1993-02-01', '1993-03-01', '1993-04-01', '1993-06-01', '1993-07-01', '1993-08-01', '1993-09-01', '1993-10-01', '1993-11-01', '1993-12-01', '1994-01-01', '1994-02-01', '1994-03-01', '1994-04-01', '1994-06-01', '1994-07-01', '1994-08-01', '1994-09-01', '1994-10-01', '1994-11-01', '1994-12-01', '1995-01-01', '1995-02-01', '1995-03-01', '1995-04-01', '1995-06-01', '1995-07-01', '1995-08-01', '1995-09-01', '1995-10-01', '1995-11-01', '1995-12-01', '1996-01-01', '1996-02-01', '1996-03-01', '1996-04-01', '2013-10-01', '2019-01-01', '2025-10-01']


In [15]:
# let's make sure that xls data made it into the main dataframe
df.query(" @pd.to_datetime(time) >= @df_xls.time.min() and @pd.to_datetime(time) <= @df_xls.time.max() ").head(25)


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
233,2010-10-01,9.47,9.15,11.04,11.04,7.57,7.53,10.77,10.77,813.0,777.0,839.0,841.0
234,2010-11-01,9.47,9.15,11.04,11.04,7.57,7.53,10.77,10.77,813.0,777.0,841.0,821.0
235,2010-12-01,9.47,9.15,11.04,11.04,7.57,7.53,10.77,10.77,813.0,777.0,821.0,814.0
236,2011-01-01,9.47,9.15,11.04,10.97,7.57,7.53,10.77,10.71,813.0,777.0,814.0,821.0
237,2011-02-01,9.47,9.15,10.97,10.97,7.57,7.53,10.71,10.71,813.0,777.0,821.0,821.0
238,2011-03-01,9.47,9.15,10.97,10.97,7.57,7.53,10.71,10.71,813.0,777.0,821.0,821.0
239,2011-04-01,9.47,9.15,10.97,10.97,7.57,7.53,10.71,10.71,813.0,777.0,821.0,811.0
240,2011-05-01,9.15,10.97,NaN,12.57,7.53,10.70,NaN,10.80,777.0,812.0,NaN,800.0
241,2011-06-01,9.15,10.97,12.57,12.57,7.53,10.70,10.80,10.20,777.0,812.0,800.0,800.0
242,2011-07-01,9.15,10.97,12.57,13.73,7.53,10.70,10.20,9.60,777.0,812.0,800.0,800.0


In [16]:
df.query(" time.isin(@known_missing_dates)  ")


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
269,2013-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
332,2019-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
413,2025-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Save output

In [17]:
# df.to_excel(f'{OUTPUT_DIR}/{CROP}_1991_2026.xlsx', index=False)
